[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/09-system-design/10_search_reliability.ipynb)

*System Design · Notebook 10 of 10*

# Search, Storage, Reliability & the Interview

The last building blocks and how to put it all together: inverted indexes, AND queries, TF-IDF, autocomplete, top-K, geohashes and nearby search, file chunking and dedupe, resumable and multipart uploads, erasure coding, SLOs, error budgets and burn rates, percentiles, retry storms, load shedding, disaster recovery, canaries — and the 45-minute interview framework.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** An inverted index is the index at the back of a book: look up the word, get the pages — no reading every page.

**What interviewers really test here:** Inverted indexes and ranking, autocomplete, geohash, chunked file storage, SLOs and error budgets, retry storms, and the interview framework.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Search and autocomplete *(Basics)*
- [ ] 2. Location and file storage *(Core)*
- [ ] 3. Reliability and operations *(Core)*
- [ ] 4. The 45-minute system design interview *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Classic questions → building blocks

| Question | Key ideas |
| --- | --- |
| URL shortener | base62, ID generation, cache, 302 redirects |
| Rate limiter | token bucket / sliding window in Redis, gateway |
| News feed | hybrid fan-out, ranking, feed cache |
| Chat | WebSockets, session registry, per-conversation seq |
| Notifications | queues, preferences, retries, digests |
| Autocomplete | trie / prefix index, top-K, batch updates |
| Web crawler | URL frontier queue, politeness, dedupe with Bloom filters |
| Ride hailing / nearby | geohash or quadtree, in-memory location updates |
| File storage | chunking, dedupe, metadata DB + object storage |
| Video streaming | transcoding pipeline, CDN, adaptive bitrate |
| Payments | idempotency, ledger, sagas, reconciliation |
| Distributed cache / KV store | consistent hashing, replication, quorums |

### How many nines?

| Availability | Downtime / month |
| --- | --- |
| 99% | 7.3 hours |
| 99.9% | 43.8 minutes |
| 99.99% | 4.4 minutes |
| 99.999% | 26 seconds |

---
# Part 2: Lessons, step by step

## Lesson 1: Search and autocomplete

*Level: Basics*

Databases can't do fast full-text search with `LIKE '%shoe%'`. A search engine (Elasticsearch, OpenSearch, Solr) builds an **inverted index**: each term → the sorted list of documents containing it. A query intersects posting lists, then ranks matches (TF-IDF, BM25, plus signals like popularity, or a learned model). Index updates flow from the database through a queue or CDC, so search is eventually consistent.



**Autocomplete** precomputes top suggestions per prefix (a trie or sorted list with cached results), updated from search logs in batches; **trending** uses top-K counting over a sliding window.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
from collections import defaultdict


def index_demo():
    docs = {1: "red shoes", 2: "red dress", 3: "running shoes"}
    index = defaultdict(list)
    for i, text in docs.items():
        for w in text.split():
            index[w].append(i)
    print(dict(index))
    print("red AND shoes →", sorted(set(index["red"]) & set(index["shoes"])))


index_demo()

{'red': [1, 2], 'shoes': [1, 3], 'dress': [2], 'running': [3]}
red AND shoes → [1]


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q12](#q12), [Q13](#q13), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 2: Location and file storage

*Level: Core*

**Nearby search** (ride hailing, food delivery, maps): bucket locations by **geohash** or a grid/quadtree/S2 cell, look up your cell and its neighbours, then compute exact distances. Moving drivers update their cell in an in-memory store every few seconds.



**File storage** (Dropbox, Google Drive, S3): metadata in a database, bytes in object storage. Split files into chunks identified by hash — upload only changed chunks, deduplicate identical ones, and resume interrupted uploads. Object stores keep data durable with replication or cheaper **erasure coding**.

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q15](#q15), [Q16](#q16), [Q4](#q4), [Q5](#q5), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 3: Reliability and operations

*Level: Core*

- **SLI** (what you measure: success rate, p99 latency) → **SLO** (target: 99.9%) → **error budget** (0.1% = 43 minutes a month). Spend the budget on releases; when it's gone, freeze risky changes.
- Alert on **burn rate**, not on every error; watch the four golden signals: latency, traffic, errors, saturation.
- Percentiles, not averages: p99 is what your heaviest users feel.
- Prevent cascades: timeouts, retries only at one layer with backoff, circuit breakers, bulkheads, **load shedding** by priority.
- Plan disasters with **RPO** (data loss allowed) and **RTO** (time to recover); release with canaries and automatic rollback.

**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q17](#q17), [Q8](#q8), [Q18](#q18), [Q19](#q19), [Q9](#q9), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 4: The 45-minute system design interview

*Level: Advanced*

1. **Requirements (5 min)**: functional features, non-functional (scale, latency, availability, consistency), what's out of scope.
2. **Estimates (3–4 min)**: QPS, peak, storage, bandwidth — only what changes the design.
3. **API and data model (5 min)**: endpoints, main tables or keys, the access patterns.
4. **High-level design (10 min)**: clients → load balancer → services → cache → databases, queues, storage. Walk one request end to end.
5. **Deep dives (15 min)**: the hardest parts — scaling the hot path, sharding, consistency, failure handling. Offer trade-offs, then choose.
6. **Wrap up (4–5 min)**: bottlenecks, monitoring, what you'd do with more time.



Practise the classics with this course's building blocks: URL shortener, rate limiter, news feed, chat, notification service, search autocomplete, web crawler, ride hailing, file storage, video streaming, payment system, distributed cache.

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q11](#q11). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
from collections import Counter, defaultdict

import heapq


<a id="q1"></a>
### Q1. Build an Inverted Index

🟢 Warm-up · *The heart of search*

Lower-case each document, split on spaces, and map every term to the sorted list of document ids containing it. Return the index with terms sorted.

**Examples**

- `docs = {1: 'Cheap red shoes', 2: 'red dress', 3: 'running shoes for men', 4: 'Red running shoes'}` → `{'cheap': [1], 'dress': [2], 'for': [3], 'men': [3], 'red': [1, 2, 4], 'running': [3, 4], 'shoes': [1, 3, 4]}`

In [ ]:
def inverted_index(docs):
    """
    Lower-case each document, split on spaces, and map every term to the sorted
    list of document ids containing it. Return the index with terms sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(inverted_index, [
    (({1: 'Cheap red shoes', 2: 'red dress', 3: 'running shoes for men', 4: 'Red running shoes'},), {'cheap': [1], 'dress': [2], 'for': [3], 'men': [3], 'red': [1, 2, 4], 'running': [3, 4], 'shoes': [1, 3, 4]}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each term, scan every document.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One pass with `defaultdict(set)`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** For each term, scan every document.

```python
def inverted_index(docs):
    terms = sorted(set(w for text in docs.values() for w in text.lower().split()))
    index = {}
    for t in terms:
        index[t] = sorted(i for i, text in docs.items() if t in text.lower().split())
    return index
```

**Cost:** O(t · d · L) time, O(t · d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One pass with `defaultdict(set)`.

```python
def inverted_index(docs):
    index = defaultdict(set)
    for i, text in docs.items():
        for w in text.lower().split():
            index[w].add(i)
    return {t: sorted(ids) for t, ids in sorted(index.items())}
```

**Cost:** O(d · L) time, O(t · d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each term, scan every document. That costs O(t · d · L) time and O(t · d) space."

"I can do better: One pass with `defaultdict(set)`. That's O(d · L) time and O(t · d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/build-an-inverted-index)

<a id="q2"></a>
### Q2. Intersect Posting Lists

🟢 Warm-up · *AND queries*

Return the doc ids present in both sorted posting lists. Search engines walk both lists with two pointers (and skip pointers) instead of building sets.

**Examples**

- `a = [1, 3, 4, 7, 9, 12, 15, 20], b = [2, 3, 7, 8, 15, 21]` → `[3, 7, 15]`

In [ ]:
def intersect(a, b):
    """
    Return the doc ids present in both sorted posting lists. Search engines walk
    both lists with two pointers (and skip pointers) instead of building sets.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(intersect, [
    (([1, 3, 4, 7, 9, 12, 15, 20], [2, 3, 7, 8, 15, 21]), [3, 7, 15]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each id of one list against the other.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two pointers.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check each id of one list against the other.

```python
def intersect(a, b):
    return [x for x in a if x in b]
```

**Cost:** O(a · b) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two pointers.

```python
def intersect(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            out.append(a[i])
            i, j = i + 1, j + 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return out
```

**Cost:** O(a + b) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each id of one list against the other. That costs O(a · b) time and O(1) space."

"I can do better: Two pointers. That's O(a + b) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/intersect-posting-lists)

<a id="q3"></a>
### Q3. Top-K Trending Searches

🟢 Warm-up · *Heavy hitters*

Return the `k` most frequent search terms in the stream with counts (ties alphabetical).

**Examples**

- `stream = ['ipl', 'gold', 'ipl', 'rain', 'ipl', 'gold', 'budget', 'rain', 'rain', 'gold', 'ipl'], k = 3` → `[('ipl', 4), ('gold', 3), ('rain', 3)]`

In [ ]:
def top_k_terms(stream, k):
    """
    Return the k most frequent search terms in the stream with counts (ties
    alphabetical).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_k_terms, [
    ((['ipl', 'gold', 'ipl', 'rain', 'ipl', 'gold', 'budget', 'rain', 'rain', 'gold', 'ipl'], 3), [('ipl', 4), ('gold', 3), ('rain', 3)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count and fully sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Counter and `heapq.nsmallest` with k.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count and fully sort.

```python
def top_k_terms(stream, k):
    counts = {}
    for s in stream:
        counts[s] = counts.get(s, 0) + 1
    return sorted(counts.items(), key=lambda x: (-x[1], x[0]))[:k]
```

**Cost:** O(n + u log u) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Counter and `heapq.nsmallest` with k.

```python
def top_k_terms(stream, k):
    return heapq.nsmallest(k, Counter(stream).items(), key=lambda x: (-x[1], x[0]))
```

**Cost:** O(n + u log k) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count and fully sort. That costs O(n + u log u) time and O(u) space."

"I can do better: Counter and `heapq.nsmallest` with k. That's O(n + u log k) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/top-k-trending-searches)

<a id="q4"></a>
### Q4. Resume an Interrupted Upload

🟢 Warm-up · *Missing chunk ranges*

A file has chunks 0..n-1 and the server has received the listed ones. Return the missing chunks as inclusive `(start, end)` ranges so the client re-sends only those.

**Examples**

- `n = 12, received = [0, 1, 2, 5, 6, 9]` → `[(3, 4), (7, 8), (10, 11)]`

In [ ]:
def missing_ranges(n, received):
    """
    A file has chunks 0..n-1 and the server has received the listed ones. Return
    the missing chunks as inclusive (start, end) ranges so the client re-sends
    only those.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(missing_ranges, [
    ((12, [0, 1, 2, 5, 6, 9]), [(3, 4), (7, 8), (10, 11)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk 0..n-1 building ranges.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same walk over a set.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk 0..n-1 building ranges.

```python
def missing_ranges(n, received):
    out, start = [], None
    for i in range(n):
        if i not in received:
            if start is None:
                start = i
        elif start is not None:
            out.append((start, i - 1))
            start = None
    if start is not None:
        out.append((start, n - 1))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same walk over a set.

```python
def missing_ranges(n, received):
    got, out, i = set(received), [], 0
    while i < n:
        if i in got:
            i += 1
            continue
        j = i
        while j + 1 < n and j + 1 not in got:
            j += 1
        out.append((i, j))
        i = j + 1
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk 0..n-1 building ranges. That costs O(n) time and O(n) space."

"I can do better: Same walk over a set. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/resume-an-interrupted-upload)

<a id="q5"></a>
### Q5. Pick a Multipart Part Size

🟢 Warm-up · *Object storage limits*

Object stores allow at most 10,000 parts of at least 5 MB. For each file size (MB), use the smallest whole-MB part size ≥ 5 that keeps parts ≤ 10,000. Return `(part MB, parts)`.

**Examples**

- `files_mb = [3, 100, 49000, 200000, 5000000]` → `[(5, 1), (5, 20), (5, 9800), (20, 10000), (500, 10000)]`

In [ ]:
def part_size(files_mb):
    """
    Object stores allow at most 10,000 parts of at least 5 MB. For each file
    size (MB), use the smallest whole-MB part size ≥ 5 that keeps parts ≤
    10,000. Return (part MB, parts).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(part_size, [
    (([3, 100, 49000, 200000, 5000000],), [(5, 1), (5, 20), (5, 9800), (20, 10000), (500, 10000)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Increase the part size until the count fits.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Ceiling division.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Increase the part size until the count fits.

```python
def part_size(files_mb):
    out = []
    for size in files_mb:
        part = 5
        while math.ceil(size / part) > 10_000:
            part += 1
        out.append((part, math.ceil(size / part)))
    return out
```

**Cost:** O(n · s) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Ceiling division.

```python
def part_size(files_mb):
    out = []
    for size in files_mb:
        part = max(5, math.ceil(size / 10_000))
        out.append((part, math.ceil(size / part)))
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Increase the part size until the count fits. That costs O(n · s) time and O(1) space."

"I can do better: Ceiling division. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/pick-a-multipart-part-size)

<a id="q6"></a>
### Q6. Replication vs Erasure Coding

🟢 Warm-up · *Durability per byte*

Store `data_tb`. Return `(raw TB, failures tolerated)` for 3× replication and for each Reed-Solomon (k data, m parity) scheme in `schemes` (raw = data × (k + m) / k).

**Examples**

- `data_tb = 100, schemes = [(6, 3), (10, 4)]` → `[(300, 2), (150.0, 3), (140.0, 4)]`

In [ ]:
def erasure(data_tb, schemes):
    """
    Store data_tb. Return (raw TB, failures tolerated) for 3× replication and
    for each Reed-Solomon (k data, m parity) scheme in schemes (raw = data × (k
    + m) / k).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(erasure, [
    ((100, [(6, 3), (10, 4)]), [(300, 2), (150.0, 3), (140.0, 4)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over the schemes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over the schemes.

```python
def erasure(data_tb, schemes):
    out = [(data_tb * 3, 2)]
    for k, m in schemes:
        out.append((round(data_tb * (k + m) / k, 1), m))
    return out
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def erasure(data_tb, schemes):
    return [(data_tb * 3, 2)] + [(round(data_tb * (k + m) / k, 1), m) for k, m in schemes]
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over the schemes. That costs O(s) time and O(s) space."

"I can do better: A comprehension. That's O(s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/replication-vs-erasure-coding)

<a id="q7"></a>
### Q7. SLOs and Error Budgets

🟢 Warm-up · *How much downtime may we spend?*

For an availability SLO (%) over `days`, return `(budget minutes, minutes left after `used`, % of budget burned)`, rounded to 1.

**Examples**

- `slo = 99.9, days = 30, used = 20` → `(43.2, 23.2, 46.3)`
- `slo = 99.99, days = 30, used = 5` → `(4.3, -0.7, 115.7)`

In [ ]:
def error_budget(slo, days, used):
    """
    For an availability SLO (%) over days, return (budget minutes, minutes left
    after used, % of budget burned), rounded to 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(error_budget, [
    ((99.9, 30, 20), (43.2, 23.2, 46.3)),
    ((99.99, 30, 5), (4.3, -0.7, 115.7)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Arithmetic step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compact.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Arithmetic step by step.

```python
def error_budget(slo, days, used):
    total = days * 24 * 60
    budget = total * (100 - slo) / 100
    left = budget - used
    burned = used / budget * 100
    return (round(budget, 1), round(left, 1), round(burned, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compact.

```python
def error_budget(slo, days, used):
    budget = days * 1440 * (1 - slo / 100)
    return (round(budget, 1), round(budget - used, 1), round(used / budget * 100, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Arithmetic step by step. That costs O(1) time and O(1) space."

"I can do better: Same, compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/slos-and-error-budgets)

<a id="q8"></a>
### Q8. Latency Percentiles

🟢 Warm-up · *Averages hide the tail*

Return the mean (rounded to 1) and the nearest-rank p50, p95 and p99 of the latencies (ms). One slow request barely moves the median but dominates p99.

**Examples**

- `lat = [12, 15, 11, 13, 900, 14, 12, 16, 13, 12, 11, 14, 15, 13, 12, 18, 13, 12, 14, 450]` → `(79.5, 13, 450, 900)`

In [ ]:
def percentiles(lat):
    """
    Return the mean (rounded to 1) and the nearest-rank p50, p95 and p99 of the
    latencies (ms). One slow request barely moves the median but dominates p99.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(percentiles, [
    (([12, 15, 11, 13, 900, 14, 12, 16, 13, 12, 11, 14, 15, 13, 12, 18, 13, 12, 14, 450],), (79.5, 13, 450, 900)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort, then index.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, via a helper.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort, then index.

```python
def percentiles(lat):
    s = sorted(lat)
    n = len(s)
    out = [round(sum(s) / n, 1)]
    for p in (50, 95, 99):
        rank = math.ceil(p / 100 * n)
        out.append(s[rank - 1])
    return tuple(out)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, via a helper.

```python
def percentiles(lat):
    s = sorted(lat)
    pick = lambda p: s[math.ceil(p / 100 * len(s)) - 1]
    return (round(sum(s) / len(s), 1), pick(50), pick(95), pick(99))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort, then index. That costs O(n log n) time and O(n) space."

"I can do better: Same, via a helper. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/latency-percentiles)

<a id="q9"></a>
### Q9. Which Disaster-Recovery Plan Fits?

🟢 Warm-up · *RPO and RTO*

Each plan has `(name, rpo_min, rto_min, monthly_cost)`: RPO = data you can lose, RTO = time to recover. Return the plans meeting both targets, cheapest first.

**Examples**

- `plans = [('backup & restore', 1440, 480, 100), ('pilot light', 15, 60, 800), ('warm standby', 1, 15, 3000), ('active-active', 0, 0, 9000)], rpo = 15, rto = 60` → `['pilot light', 'warm standby', 'active-active']`
- `plans = [('backup & restore', 1440, 480, 100), ('pilot light', 15, 60, 800), ('warm standby', 1, 15, 3000), ('active-active', 0, 0, 9000)], rpo = 0, rto = 5` → `['active-active']`

In [ ]:
def dr_plans(plans, rpo, rto):
    """
    Each plan has (name, rpo_min, rto_min, monthly_cost): RPO = data you can
    lose, RTO = time to recover. Return the plans meeting both targets, cheapest
    first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dr_plans, [
    (([('backup & restore', 1440, 480, 100), ('pilot light', 15, 60, 800), ('warm standby', 1, 15, 3000), ('active-active', 0, 0, 9000)], 15, 60), ['pilot light', 'warm standby', 'active-active']),
    (([('backup & restore', 1440, 480, 100), ('pilot light', 15, 60, 800), ('warm standby', 1, 15, 3000), ('active-active', 0, 0, 9000)], 0, 5), ['active-active']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Filter in a loop, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A sorted comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Filter in a loop, then sort.

```python
def dr_plans(plans, rpo, rto):
    ok = []
    for name, p_rpo, p_rto, cost in plans:
        if p_rpo <= rpo and p_rto <= rto:
            ok.append((cost, name))
    ok.sort()
    return [name for _, name in ok]
```

**Cost:** O(p log p) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A sorted comprehension.

```python
def dr_plans(plans, rpo, rto):
    return [p[0] for p in sorted((p for p in plans if p[1] <= rpo and p[2] <= rto), key=lambda p: p[3])]
```

**Cost:** O(p log p) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Filter in a loop, then sort. That costs O(p log p) time and O(p) space."

"I can do better: A sorted comprehension. That's O(p log p) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/which-disaster-recovery-plan-fits)

<a id="q10"></a>
### Q10. Budget Your Interview Time

🟢 Warm-up · *The framework, in minutes*

Split an interview of `minutes` across `PHASES` by weight. Return `(phase, minutes rounded, minute it ends at)`.

**Examples**

- `minutes = 45` → `[('requirements', 5, 5), ('estimation', 4, 9), ('api & data model', 5, 14), ('high-level design', 11, 26), ('deep dives', 15, 40), ('wrap up', 4, 45)]`
- `minutes = 60` → `[('requirements', 7, 7), ('estimation', 5, 12), ('api & data model', 7, 19), ('high-level design', 15, 34), ('deep dives', 20, 54), ('wrap up', 6, 60)]`

In [ ]:
PHASES = [("requirements", 0.12), ("estimation", 0.08), ("api & data model", 0.12),
          ("high-level design", 0.25), ("deep dives", 0.33), ("wrap up", 0.10)]

def interview_plan(minutes):
    """
    Split an interview of minutes across PHASES by weight. Return (phase,
    minutes rounded, minute it ends at).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(interview_plan, [
    ((45,), [('requirements', 5, 5), ('estimation', 4, 9), ('api & data model', 5, 14), ('high-level design', 11, 26), ('deep dives', 15, 40), ('wrap up', 4, 45)]),
    ((60,), [('requirements', 7, 7), ('estimation', 5, 12), ('api & data model', 7, 19), ('high-level design', 15, 34), ('deep dives', 20, 54), ('wrap up', 6, 60)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with a running total.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.accumulate`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with a running total.

```python
def interview_plan(minutes):
    out, t = [], 0
    for name, w in PHASES:
        t += w * minutes
        out.append((name, round(w * minutes), round(t)))
    return out
```

**Cost:** O(p) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.accumulate`.

```python
def interview_plan(minutes):
    from itertools import accumulate
    ends = accumulate(w * minutes for _, w in PHASES)
    return [(name, round(w * minutes), round(e)) for (name, w), e in zip(PHASES, ends)]
```

**Cost:** O(p) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with a running total. That costs O(p) time and O(p) space."

"I can do better: `itertools.accumulate`. That's O(p) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/budget-your-interview-time)

<a id="q11"></a>
### Q11. Self-Review a Design Answer

🟢 Warm-up · *Did you cover everything?*

Given the topics a practice answer covered, return the `CHECKLIST` items still missing (in checklist order) and the coverage %, rounded to 1.

**Examples**

- `covered = ['api', 'high-level diagram', 'functional requirements', 'scaling', 'data model', 'caching']` → `(['non-functional requirements', 'estimates', 'bottlenecks', 'failure handling', 'monitoring'], 50.0)`

In [ ]:
CHECKLIST = ["functional requirements", "non-functional requirements", "estimates", "api", "data model",
             "high-level diagram", "scaling", "bottlenecks", "failure handling", "monitoring"]

def design_review(covered):
    """
    Given the topics a practice answer covered, return the CHECKLIST items still
    missing (in checklist order) and the coverage %, rounded to 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(design_review, [
    ((['api', 'high-level diagram', 'functional requirements', 'scaling', 'data model', 'caching'],), (['non-functional requirements', 'estimates', 'bottlenecks', 'failure handling', 'monitoring'], 50.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each item against the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of covered topics.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check each item against the list.

```python
def design_review(covered):
    missing = []
    for item in CHECKLIST:
        if item not in covered:
            missing.append(item)
    return (missing, round((len(CHECKLIST) - len(missing)) / len(CHECKLIST) * 100, 1))
```

**Cost:** O(c · a) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of covered topics.

```python
def design_review(covered):
    have = set(covered)
    missing = [c for c in CHECKLIST if c not in have]
    return (missing, round(100 * (1 - len(missing) / len(CHECKLIST)), 1))
```

**Cost:** O(c + a) time, O(a) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each item against the list. That costs O(c · a) time and O(c) space."

"I can do better: A set of covered topics. That's O(c + a) time and O(a) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/self-review-a-design-answer)

<a id="q12"></a>
### Q12. Rank Results With TF-IDF

🟡 Interview standard · *Rare words matter more*

Score each document for the query terms as Σ tf × log(N / df) (tf = count of the term in the doc, df = docs containing it). Return `(doc id, score rounded to 3)` for documents with a positive score, best first (ties by id).

**Examples**

- `docs = {1: 'red shoes red', 2: 'blue shoes', 3: 'red dress', 4: 'running shoes'}, query = 'red shoes'` → `[(1, 1.674), (3, 0.693), (2, 0.288), (4, 0.288)]`

In [ ]:
def tfidf(docs, query):
    """
    Score each document for the query terms as Σ tf × log(N / df) (tf = count of
    the term in the doc, df = docs containing it). Return (doc id, score rounded
    to 3) for documents with a positive score, best first (ties by id).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tfidf, [
    (({1: 'red shoes red', 2: 'blue shoes', 3: 'red dress', 4: 'running shoes'}, 'red shoes'), [(1, 1.674), (3, 0.693), (2, 0.288), (4, 0.288)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count terms by scanning text for every query term and doc.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Precompute term counts and document frequencies.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count terms by scanning text for every query term and doc.

```python
def tfidf(docs, query):
    n = len(docs)
    scores = []
    for i, text in docs.items():
        words = text.split()
        s = 0
        for t in query.split():
            tf = words.count(t)
            df = sum(1 for other in docs.values() if t in other.split())
            if tf and df:
                s += tf * math.log(n / df)
        if s > 0:
            scores.append((i, round(s, 3)))
    scores.sort(key=lambda x: (-x[1], x[0]))
    return scores
```

**Cost:** O(q · d · L) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Precompute term counts and document frequencies.

```python
def tfidf(docs, query):
    counts = {i: Counter(t.split()) for i, t in docs.items()}
    df = Counter(w for c in counts.values() for w in c)
    idf = {t: math.log(len(docs) / df[t]) for t in query.split() if df[t]}
    scores = [(i, round(sum(c[t] * w for t, w in idf.items()), 3)) for i, c in counts.items()]
    return sorted([s for s in scores if s[1] > 0], key=lambda x: (-x[1], x[0]))
```

**Cost:** O(d · L + q · d) time, O(d · L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count terms by scanning text for every query term and doc. That costs O(q · d · L) time and O(d) space."

"I can do better: Precompute term counts and document frequencies. That's O(d · L + q · d) time and O(d · L) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/rank-results-with-tf-idf)

<a id="q13"></a>
### Q13. Autocomplete

🟡 Interview standard · *Top suggestions for a prefix*

Given past search counts, return the `k` most searched queries starting with each prefix (ties alphabetical).

**Examples**

- `counts = {'iphone': 900, 'iphone case': 400, 'ipad': 700, 'ice cream': 300, 'iphone charger': 400, 'idli': 50}, prefixes = ['ip', 'iphone ', 'i', 'x'], k = 3` → `[['iphone', 'ipad', 'iphone case'], ['iphone case', 'iphone charger'], ['iphone', 'ipad', 'iphone case'], []]`

In [ ]:
import bisect

def autocomplete(counts, prefixes, k):
    """
    Given past search counts, return the k most searched queries starting with
    each prefix (ties alphabetical).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(autocomplete, [
    (({'iphone': 900, 'iphone case': 400, 'ipad': 700, 'ice cream': 300, 'iphone charger': 400, 'idli': 50}, ['ip', 'iphone ', 'i', 'x'], 3), [['iphone', 'ipad', 'iphone case'], ['iphone case', 'iphone charger'], ['iphone', 'ipad', 'iphone case'], []]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan all queries for each prefix, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort once; `bisect` to the prefix range.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan all queries for each prefix, then sort.

```python
def autocomplete(counts, prefixes, k):
    out = []
    for p in prefixes:
        matches = [(-c, q) for q, c in counts.items() if q.startswith(p)]
        matches.sort()
        out.append([q for _, q in matches[:k]])
    return out
```

**Cost:** O(p · n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sort once; `bisect` to the prefix range.

```python
def autocomplete(counts, prefixes, k):
    keys = sorted(counts)
    out = []
    for p in prefixes:
        lo, hi = bisect.bisect_left(keys, p), bisect.bisect_left(keys, p + "￿")
        out.append(heapq.nsmallest(k, keys[lo:hi], key=lambda q: (-counts[q], q)))
    return out
```

**Cost:** O(n log n + p · r log r) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan all queries for each prefix, then sort. That costs O(p · n log n) time and O(n) space."

"I can do better: Sort once; `bisect` to the prefix range. That's O(n log n + p · r log r) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/autocomplete)

<a id="q14"></a>
### Q14. Encode a Geohash

🟡 Interview standard · *Locations as sortable strings*

Encode each `(lat, lon)` as a geohash of `precision` characters: alternately halve the longitude and latitude ranges (longitude first), writing 1 for the upper half, and map each 5 bits to base32. Nearby places share prefixes.

**Examples**

- `points = [(28.6139, 77.209), (28.6129, 77.2295), (19.076, 72.8777), (51.5074, -0.1278)], precision = 6` → `['ttnfuc', 'ttnfv2', 'te7ud2', 'gcpvj0']`

In [ ]:
BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"

def geohash(points, precision):
    """
    Encode each (lat, lon) as a geohash of precision characters: alternately
    halve the longitude and latitude ranges (longitude first), writing 1 for the
    upper half, and map each 5 bits to base32. Nearby places share prefixes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(geohash, [
    (([(28.6139, 77.209), (28.6129, 77.2295), (19.076, 72.8777), (51.5074, -0.1278)], 6), ['ttnfuc', 'ttnfv2', 'te7ud2', 'gcpvj0']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a list of bits, then convert.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Accumulate 5-bit integers directly.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Build a list of bits, then convert.

```python
def geohash(points, precision):
    out = []
    for lat, lon in points:
        lat_r, lon_r = [-90.0, 90.0], [-180.0, 180.0]
        bits = []
        even = True
        while len(bits) < precision * 5:
            rng, val = (lon_r, lon) if even else (lat_r, lat)
            mid = (rng[0] + rng[1]) / 2
            if val >= mid:
                bits.append(1)
                rng[0] = mid
            else:
                bits.append(0)
                rng[1] = mid
            even = not even
        s = ""
        for i in range(0, len(bits), 5):
            s += BASE32[int("".join(map(str, bits[i:i + 5])), 2)]
        out.append(s)
    return out
```

**Cost:** O(n · p) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Accumulate 5-bit integers directly.

```python
def geohash(points, precision):
    def enc(lat, lon):
        rngs, vals, chars, bit, ch, even = [[-180.0, 180.0], [-90.0, 90.0]], [lon, lat], [], 0, 0, 0
        while len(chars) < precision:
            r, v = rngs[even], vals[even]
            mid = (r[0] + r[1]) / 2
            up = v >= mid
            ch = ch << 1 | up
            r[0 if up else 1] = mid
            even, bit = 1 - even, bit + 1
            if bit == 5:
                chars.append(BASE32[ch])
                bit = ch = 0
        return "".join(chars)
    return [enc(lat, lon) for lat, lon in points]
```

**Cost:** O(n · p) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a list of bits, then convert. That costs O(n · p) time and O(p) space."

"I can do better: Accumulate 5-bit integers directly. That's O(n · p) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/encode-a-geohash)

<a id="q15"></a>
### Q15. Find Nearby Places

🟡 Interview standard · *Grid cells instead of scanning everything*

Return `(name, km rounded to 2)` for places within `radius_km` of `me`, nearest first. The optimal version buckets places into grid cells of `cell` degrees and only checks the 9 cells around you (valid while the radius is smaller than a cell).

**Examples**

- `places = {'cafe': (28.6145, 77.21), 'mall': (28.63, 77.22), 'park': (28.6, 77.25), 'temple': (28.6129, 77.2295), 'airport': (28.5562, 77.1)}, me = (28.6139, 77.209), radius_km = 3.0, cell = 0.05` → `[('cafe', 0.12), ('temple', 2.0), ('mall', 2.09)]`

In [ ]:
def nearby(places, me, radius_km, cell):
    """
    Return (name, km rounded to 2) for places within radius_km of me, nearest
    first. The optimal version buckets places into grid cells of cell degrees
    and only checks the 9 cells around you (valid while the radius is smaller
    than a cell).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(nearby, [
    (({'cafe': (28.6145, 77.21), 'mall': (28.63, 77.22), 'park': (28.6, 77.25), 'temple': (28.6129, 77.2295), 'airport': (28.5562, 77.1)}, (28.6139, 77.209), 3.0, 0.05), [('cafe', 0.12), ('temple', 2.0), ('mall', 2.09)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Haversine distance to every place.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Grid buckets, then 9 cells.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Haversine distance to every place.

```python
def km(a, b):
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(h))


def nearby(places, me, radius_km, cell):
    out = []
    for name, p in places.items():
        d = km(me, p)
        if d <= radius_km:
            out.append((round(d, 2), name))
    out.sort()
    return [(n, d) for d, n in out]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Grid buckets, then 9 cells.

```python
def km(a, b):
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(h))


def nearby(places, me, radius_km, cell):
    grid = defaultdict(list)
    for name, (lat, lon) in places.items():
        grid[int(lat // cell), int(lon // cell)].append(name)
    ci, cj = int(me[0] // cell), int(me[1] // cell)
    found = []
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            for name in grid.get((ci + di, cj + dj), []):
                d = km(me, places[name])
                if d <= radius_km:
                    found.append((round(d, 2), name))
    return [(n, d) for d, n in sorted(found)]
```

**Cost:** O(n) build, O(nearby) query time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Haversine distance to every place. That costs O(n) time and O(1) space."

"I can do better: Grid buckets, then 9 cells. That's O(n) build, O(nearby) query time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/find-nearby-places)

<a id="q16"></a>
### Q16. Chunk and Deduplicate Files

🟡 Interview standard · *Upload only what changed*

Split each file version into chunks of `size` characters and identify chunks by SHA-256. Return `(chunks in v2, chunks the client must upload)` given the server already stores v1's chunks (Dropbox-style sync).

**Examples**

- `v1 = 'the quick brown fox jumps over the lazy dog!', v2 = 'the quick brown cat jumps over the lazy dog!', size = 8` → `(6, 1)`

In [ ]:
import hashlib

def chunk_dedupe(v1, v2, size):
    """
    Split each file version into chunks of size characters and identify chunks
    by SHA-256. Return (chunks in v2, chunks the client must upload) given the
    server already stores v1's chunks (Dropbox-style sync).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(chunk_dedupe, [
    (('the quick brown fox jumps over the lazy dog!', 'the quick brown cat jumps over the lazy dog!', 8), (6, 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare chunk strings against a list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of chunk hashes.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare chunk strings against a list.

```python
def chunk_dedupe(v1, v2, size):
    old = [v1[i:i + size] for i in range(0, len(v1), size)]
    new = [v2[i:i + size] for i in range(0, len(v2), size)]
    upload = []
    for c in new:
        if c not in old and c not in upload:
            upload.append(c)
    return (len(new), len(upload))
```

**Cost:** O(c²) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of chunk hashes.

```python
def chunk_dedupe(v1, v2, size):
    def hashes(s):
        return [hashlib.sha256(s[i:i + size].encode()).hexdigest() for i in range(0, len(s), size)]
    old, new = set(hashes(v1)), hashes(v2)
    return (len(new), len(set(new) - old))
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare chunk strings against a list. That costs O(c²) time and O(c) space."

"I can do better: A set of chunk hashes. That's O(c) time and O(c) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `v1 = 'aaaabbbb', v2 = 'aaaacccc', size = 4`

1. hashes() returns ['61be55a8e2f6b4e172338bddf184d6dbee2...
2. hashes() returns ['61be55a8e2f6b4e172338bddf184d6dbee2...
3. chunk\_dedupe() returns (2, 1)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/chunk-and-deduplicate-files)

<a id="q17"></a>
### Q17. Burn-Rate Alerts

🟡 Interview standard · *Page on fast budget burn*

Burn rate = error rate / (1 − SLO). A burn rate of 14.4 over an hour spends 2% of a 30-day budget in that hour. For hourly error rates (%), return each hour's burn rate (rounded to 1) and the hours (indexes) to page on (≥ `threshold`).

**Examples**

- `error_pct = [0.05, 0.2, 1.5, 3.0, 0.1], slo = 99.9, threshold = 14.4` → `([0.5, 2.0, 15.0, 30.0, 1.0], [2, 3])`

In [ ]:
def burn_rate(error_pct, slo, threshold):
    """
    Burn rate = error rate / (1 − SLO). A burn rate of 14.4 over an hour spends
    2% of a 30-day budget in that hour. For hourly error rates (%), return each
    hour's burn rate (rounded to 1) and the hours (indexes) to page on (≥
    threshold).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(burn_rate, [
    (([0.05, 0.2, 1.5, 3.0, 0.1], 99.9, 14.4), ([0.5, 2.0, 15.0, 30.0, 1.0], [2, 3])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop.

```python
def burn_rate(error_pct, slo, threshold):
    rates, page = [], []
    for i, e in enumerate(error_pct):
        r = e / (100 - slo)
        rates.append(round(r, 1))
        if r >= threshold:
            page.append(i)
    return (rates, page)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehensions.

```python
def burn_rate(error_pct, slo, threshold):
    raw = [e / (100 - slo) for e in error_pct]
    return ([round(r, 1) for r in raw], [i for i, r in enumerate(raw) if r >= threshold])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(n) space."

"I can do better: Comprehensions. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/burn-rate-alerts)

<a id="q18"></a>
### Q18. Retry Storms

🟡 Interview standard · *Retries multiply across layers*

A request passes through layers that each retry a failed call `r` times. If the bottom service is down, one user request causes ∏(r + 1) calls to it. Return the amplification for each stack, and for the same stack if only the top layer retries.

**Examples**

- `stacks = [[2, 2, 2], [3, 3, 3, 3], [1, 0, 0]]` → `[(27, 3), (256, 4), (2, 2)]`

In [ ]:
def retry_storm(stacks):
    """
    A request passes through layers that each retry a failed call r times. If
    the bottom service is down, one user request causes ∏(r + 1) calls to it.
    Return the amplification for each stack, and for the same stack if only the
    top layer retries.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(retry_storm, [
    (([[2, 2, 2], [3, 3, 3, 3], [1, 0, 0]],), [(27, 3), (256, 4), (2, 2)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.prod`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Multiply in a loop.

```python
def retry_storm(stacks):
    out = []
    for layers in stacks:
        total = 1
        for r in layers:
            total *= r + 1
        out.append((total, layers[0] + 1))
    return out
```

**Cost:** O(s · l) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `math.prod`.

```python
def retry_storm(stacks):
    return [(math.prod(r + 1 for r in layers), layers[0] + 1) for layers in stacks]
```

**Cost:** O(s · l) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply in a loop. That costs O(s · l) time and O(s) space."

"I can do better: `math.prod`. That's O(s · l) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/retry-storms)

<a id="q19"></a>
### Q19. Shed Load by Priority

🟡 Interview standard · *Degrade gracefully*

Under overload you can serve only `capacity` requests. Keep the highest-priority ones (priority 0 = critical, e.g. checkout; higher = less important), earliest first within a priority. Return the served ids in arrival order.

**Examples**

- `requests = [('r1', 2), ('r2', 0), ('r3', 1), ('r4', 2), ('r5', 0), ('r6', 1), ('r7', 3)], capacity = 4` → `['r2', 'r3', 'r5', 'r6']`

In [ ]:
def shed(requests, capacity):
    """
    Under overload you can serve only capacity requests. Keep the highest-
    priority ones (priority 0 = critical, e.g. checkout; higher = less
    important), earliest first within a priority. Return the served ids in
    arrival order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shed, [
    (([('r1', 2), ('r2', 0), ('r3', 1), ('r4', 2), ('r5', 0), ('r6', 1), ('r7', 3)], 4), ['r2', 'r3', 'r5', 'r6']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort by (priority, arrival), take the top, re-sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`heapq.nsmallest` then sort indexes.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort by (priority, arrival), take the top, re-sort.

```python
def shed(requests, capacity):
    indexed = [(p, i, rid) for i, (rid, p) in enumerate(requests)]
    indexed.sort()
    kept = indexed[:capacity]
    kept.sort(key=lambda x: x[1])
    return [rid for _, _, rid in kept]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `heapq.nsmallest` then sort indexes.

```python
def shed(requests, capacity):
    keep = heapq.nsmallest(capacity, range(len(requests)), key=lambda i: (requests[i][1], i))
    return [requests[i][0] for i in sorted(keep)]
```

**Cost:** O(n log c) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort by (priority, arrival), take the top, re-sort. That costs O(n log n) time and O(n) space."

"I can do better: `heapq.nsmallest` then sort indexes. That's O(n log c) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/shed-load-by-priority)

<a id="q20"></a>
### Q20. Promote or Roll Back the Canary?

🟡 Interview standard · *Automated canary analysis*

Compare the canary's metrics with the baseline: roll back if its error rate exceeds the baseline's by more than `err_margin` percentage points or its p99 latency is more than `lat_ratio` times the baseline's. Return the decision and the reasons.

**Examples**

- `baseline = {'err': 0.4, 'p99': 180}, candidate = {'err': 0.5, 'p99': 190}, err_margin = 0.5, lat_ratio = 1.2` → `('promote', [])`
- `baseline = {'err': 0.4, 'p99': 180}, candidate = {'err': 1.2, 'p99': 400}, err_margin = 0.5, lat_ratio = 1.2` → `('rollback', ['errors', 'latency'])`

In [ ]:
def canary(baseline, candidate, err_margin, lat_ratio):
    """
    Compare the canary's metrics with the baseline: roll back if its error rate
    exceeds the baseline's by more than err_margin percentage points or its p99
    latency is more than lat_ratio times the baseline's. Return the decision and
    the reasons.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(canary, [
    (({'err': 0.4, 'p99': 180}, {'err': 0.5, 'p99': 190}, 0.5, 1.2), ('promote', [])),
    (({'err': 0.4, 'p99': 180}, {'err': 1.2, 'p99': 400}, 0.5, 1.2), ('rollback', ['errors', 'latency'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

If-statements collecting reasons.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A list of checks.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** If-statements collecting reasons.

```python
def canary(baseline, candidate, err_margin, lat_ratio):
    reasons = []
    if candidate["err"] - baseline["err"] > err_margin:
        reasons.append("errors")
    if candidate["p99"] > baseline["p99"] * lat_ratio:
        reasons.append("latency")
    if reasons:
        return ("rollback", reasons)
    return ("promote", reasons)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A list of checks.

```python
def canary(baseline, candidate, err_margin, lat_ratio):
    checks = [("errors", candidate["err"] - baseline["err"] > err_margin),
              ("latency", candidate["p99"] > baseline["p99"] * lat_ratio)]
    reasons = [name for name, bad in checks if bad]
    return ("rollback" if reasons else "promote", reasons)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: If-statements collecting reasons. That costs O(1) time and O(1) space."

"I can do better: A list of checks. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/search_reliability/promote-or-roll-back-the-canary)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What do interviewers look for?</b></summary>

Clarifying the problem, numbers that drive decisions, a clear working design, depth on the hard parts, explicit trade-offs, and handling of failures — communicated as a conversation, not a monologue.

</details>

<details>
<summary><b>2. How do you design search for an e-commerce site?</b></summary>

Stream product changes into a search engine's inverted index; rank by text relevance plus business signals; add filters (facets), autocomplete from query logs, and cache popular queries.

</details>

<details>
<summary><b>3. What is an error budget used for?</b></summary>

It turns reliability into a shared number: while budget remains, teams ship; when it's spent, they prioritise reliability work. It stops both over- and under-investing in reliability.

</details>

<details>
<summary><b>4. How do you find nearby drivers quickly?</b></summary>

Index driver locations by geohash or grid cell in memory, query your cell plus neighbours, compute exact distances for the few candidates, and refresh locations every few seconds.

</details>

<details>
<summary><b>5. How do you prevent a retry storm?</b></summary>

Retry at one layer only, with capped exponential backoff and jitter, retry budgets, and circuit breakers; return errors quickly when overloaded instead of queueing forever.

</details>

<details>
<summary><b>6. What should you do if you don't know something in the interview?</b></summary>

Say so, reason from first principles, state an assumption, and keep going. Interviewers value structured thinking over memorised answers.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Search and autocomplete
- [ ] I can explain: Location and file storage
- [ ] I can explain: Reliability and operations
- [ ] I can explain: The 45-minute system design interview

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Chunk and Deduplicate Files
- [ ] [Q17](#q17) Burn-Rate Alerts
- [ ] [Q18](#q18) Retry Storms
- [ ] [Q19](#q19) Shed Load by Priority
- [ ] [Q20](#q20) Promote or Roll Back the Canary?

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.